# A7: Set up a tiny lakehouse for the Pop-Tart data

**IT 630 · Friday, October 16 · guided walkthrough · about 45 minutes**

**The ask.** Dana Reyes is back. Finance trusts the audited Pop-Tart export now
(the 5,713 strawberry units from Week 1), but it still lives as a single CSV on
someone's drive. Dana wants it somewhere the whole team can query the same way,
and she has a question she wants answered from it:

> **During the storm window, September 5 through 9, which store sold the most
> strawberry Pop-Tarts, and how many units?**

Your job today is to **set that up** and answer her question from it.

**Today we build a local miniature.** Everything you create lives in your own
Colab session and disappears when it ends. A shared system would keep these
tables in durable shared storage and give the team access to them. The parts
and how they fit together are the same; only the location and the size differ.

You will build it in four small steps, each one a little closer to a lakehouse:

1. **A star:** split the export into a fact table and a dimension table (L11).
2. **A lake:** save the fact table as a folder of Parquet files (L12).
3. **A lakehouse:** save the same data as a Delta table, which is files *plus a log* (L12).
4. **The analyst question:** answer Dana from the lakehouse.

**How this works.** This is not a run-all-cells notebook. You'll see three kinds of cells:

- **▶️ Run** cells: read the comment at the top, then run them (Shift+Enter).
- **🧩 Fill-in** cells: replace each `___` before you run them. They error until you do.
- **✏️ Answer** cells: double-click, type a short answer (a sentence or two), then
  Shift+Enter. Some ask you to **predict before you run** the next cell. Write
  the prediction first; a wrong prediction is fine and is the point.

**Submit:** File → Download → Download .ipynb, and upload that file to the Canvas
activity. Before you download, make sure every 🧩 cell has run and every ✏️ cell
has an answer.

First: **File → Save a copy in Drive**, so your answers are saved.

## Step 0. Set up (about 5 minutes)

Colab already has `pandas` and `duckdb` (a SQL engine that runs inside the
notebook). It does not have `deltalake`, the library that writes Delta tables,
so the first cell installs it. That takes about 20 seconds.

In [ ]:
# ▶️ Run: install deltalake and load the tools.
%pip install -q deltalake==1.6.6

import os, json, glob, shutil
import pandas as pd
import duckdb
import pyarrow as pa
import pyarrow.parquet as pq
from deltalake import write_deltalake, DeltaTable

print("ready")

In [ ]:
# ▶️ Run: load the audited export Finance used (the same file as Week 1).
raw = pd.read_csv("https://blkfin.github.io/it630/fall2026/activities/data/poptart-sales-2026-09-10-full.csv")

print(raw.shape)
raw.head()

**✏️ Q1. One row of `raw` means one ___. Fill in the sentence. (Hint: which three columns together make a row unique?) — your answer:** _double-click this cell and type here._

## Step 1. Split the export into a star (about 10 minutes)

Every row of `raw` repeats the product's full name and flavor, even though there
are only a few products. In a star, those descriptions live once in a
**dimension** table, and the **fact** table keeps the measurements plus a key
that points to the dimension.

In [ ]:
# ▶️ Run: build the fact table and the product dimension.
fact_sales = raw[["date", "store_id", "sku", "units", "unit_price"]]
dim_product = raw[["sku", "product_name", "flavor"]].drop_duplicates().reset_index(drop=True)

print("fact_sales:", fact_sales.shape)
fact_sales.head(3)

**✏️ Q2. Predict before you run the next cell: how many rows will `dim_product` have, and why? — your answer:** _double-click this cell and type here._

In [ ]:
# ▶️ Run: look at the dimension.
dim_product

A star only works if each key appears **once** in the dimension. If `PT-STRW`
appeared twice, every strawberry sale would match twice in a join and the
storm-window total would double, from 5,713 to 11,426.

Check it. DuckDB can run SQL directly against a pandas table by its variable
name. Replace `___` with the column that is supposed to be unique.

In [ ]:
# 🧩 Fill-in: replace ___ with the dimension's key column, then run.
duckdb.sql("""
    SELECT COUNT(*)              AS rows,
           COUNT(DISTINCT ___)   AS distinct_keys
    FROM dim_product
""")

**✏️ Q3. The two numbers match. What does that tell you? Why do we run this check on `dim_product` and not on the result of joining it to `fact_sales`? — your answer:** _double-click this cell and type here._

## Step 2. A lake is a folder of files (about 8 minutes)

A data lake stores tables as plain files, usually Parquet. Big tables are split
into one folder per value of a column, so a query that only needs a few days
only opens those days' files. This is called **partitioning**. The next cell
writes `fact_sales` partitioned by `date`.

*A note on realism:* splitting 684 rows into 19 folders of 36 rows each is far
too fine for real work. Real teams partition by date only when each folder holds
a lot of data, because thousands of tiny files slow a lake down. We do it here
so you can see the structure.

In [ ]:
# ▶️ Run: write the fact table into a lake folder, one subfolder per date.
shutil.rmtree("lake", ignore_errors=True)   # start fresh, so re-running this cell is safe
pq.write_to_dataset(
    pa.Table.from_pandas(fact_sales, preserve_index=False),
    "lake/fact_sales",
    partition_cols=["date"],
)

folders = sorted(os.listdir("lake/fact_sales"))
print(len(folders), "folders")
print(folders[:3], "...")
print("inside one folder:", os.listdir("lake/fact_sales/" + folders[0]))

In [ ]:
# ▶️ Run: open ONE of those files directly and look at its columns.
one_file = glob.glob("lake/fact_sales/date=2026-09-05/*.parquet")[0]
pd.read_parquet(one_file).head(3)

**✏️ Q4. One column from `fact_sales` is missing inside that file. Which one, and where did that information go? — your answer:** _double-click this cell and type here._

In [ ]:
# ▶️ Run: DuckDB treats every file that matches the pattern as one table.
duckdb.sql("""
    SELECT COUNT(*) AS rows
    FROM read_parquet('lake/fact_sales/**/*.parquet', hive_partitioning = true)
""")

**✏️ Q5. In this lake, what decides which files count as "the table"? (Look at the `FROM` line.) — your answer:** _double-click this cell and type here._

## Step 3. A lakehouse is files plus a log (about 12 minutes)

A **Delta table** stores the same Parquet files, but adds a folder called
`_delta_log`. Every change to the table is saved as a numbered entry in that
log, called a **commit**. A reader uses the log to work out which files belong
to the version it asks for, instead of trusting whatever is in the folder. That
is the idea from Wednesday: the log is what turns a pile of files into a table.

In [ ]:
# ▶️ Run: save BOTH halves of the star as Delta tables.
shutil.rmtree("lakehouse", ignore_errors=True)   # start fresh, so re-running this cell is safe
write_deltalake("lakehouse/fact_sales", fact_sales, partition_by=["date"])
write_deltalake("lakehouse/dim_product", dim_product)

print("tables:", sorted(os.listdir("lakehouse")))
print(sorted(os.listdir("lakehouse/fact_sales"))[:4], "...")
print("log entries:", sorted(os.listdir("lakehouse/fact_sales/_delta_log")))

**✏️ Q6. Predict before you run the next cell: the log's first entry lists one `add` action for each data file. How many `add` actions will it have? — your answer:** _double-click this cell and type here._

In [ ]:
# ▶️ Run: read the log entry. Each line is one action.
log_file = "lakehouse/fact_sales/_delta_log/00000000000000000000.json"
actions = [json.loads(line) for line in open(log_file)]

for a in actions:
    kind = list(a)[0]
    detail = a[kind].get("path", a[kind].get("operation", ""))
    print(f"{kind:<11} {detail}")

In [ ]:
# ▶️ Run: ask each table what version it is and which files it is made of.
dt = DeltaTable("lakehouse/fact_sales")
dp = DeltaTable("lakehouse/dim_product")
print("fact_sales:  version", dt.version(), "|", len(dt.file_uris()), "files")
print("dim_product: version", dp.version(), "|", len(dp.file_uris()), "files")

**✏️ Q7. In your own words: what does the lakehouse know about its table that the lake in Step 2 did not? — your answer:** _double-click this cell and type here._

## Step 4. Answer Dana from the lakehouse (about 10 minutes)

Back to Dana's question: **during the storm window, September 5 through 9,
which store sold the most strawberry Pop-Tarts, and how many units?**

This time, read both tables back **from the lakehouse**, not from the notebook's
memory, the way a teammate would. Then check the total against Finance's number.

In [ ]:
# ▶️ Run: read both lakehouse tables, then total strawberry units for the window.
fact = dt.to_pandas()
product = dp.to_pandas()

duckdb.sql("""
    SELECT SUM(f.units) AS strawberry_units
    FROM fact AS f
    JOIN product AS p ON f.sku = p.sku
    WHERE p.flavor = 'Strawberry'
      AND f.date BETWEEN '2026-09-05' AND '2026-09-09'
""")

That should be **5,713**, Finance's number. Now break it down by store. Fill in
the two blanks so you get **one row per store**, largest first.

In [ ]:
# 🧩 Fill-in: replace both ___ so the result has one row per store.
duckdb.sql("""
    SELECT f.___ AS store, SUM(f.units) AS strawberry_units
    FROM fact AS f
    JOIN product AS p ON f.sku = p.sku
    WHERE p.flavor = 'Strawberry'
      AND f.date BETWEEN '2026-09-05' AND '2026-09-09'
    GROUP BY ___
    ORDER BY strawberry_units DESC
""")

**✏️ Q8. One row of your result means one ___. Then answer Dana: which store, and how many units? — your answer:** _double-click this cell and type here._

## Wrap-up (about 5 minutes)

**✏️ Q9. Dana asks which version of each table your answer came from. What do you tell her, and how do you know? — your answer:** _double-click this cell and type here._

**✏️ Q10. The log says the table is version 0 and lists every file. Does that prove the numbers inside those files are right? Name one thing the log cannot check for you. — your answer:** _double-click this cell and type here._

**Coming up.** Today the folder and the log agree. Next, we'll see what happens
when they don't: a correction gets committed, and a file lands in the folder
without ever being added to the log.